# 09. 새 HCC count 데이터에 라벨 없이 추론하기

학습된 분류 bundle을 고정해 새 세포의 broad lineage를 예측합니다. 기본 동작은 실제 개발 세포64개의 인터페이스 검증 결과 읽기입니다. 새 query를 이용한 fitting·adaptation이나 정답 라벨 입력이 없습니다. 외부 최종 cohort의 평가 독립성은 별도 freeze 계약으로 관리합니다.


In [1]:
from pathlib import Path
import json, subprocess
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
REPORT = ROOT / 'reports/research_phase2'


## 입력과 출력

입력은 cells×genes AnnData `.h5ad`입니다. `layers["counts"]`가 있으면 이를 사용하고, 없으면 `X`를 raw nonnegative integer counts로 읽습니다. 세포 index는 고유해야 하며 var에는 `ensembl_id`, `gene_symbol`이 필요합니다. 정규화값을 raw counts처럼 넣지 않습니다. 정답 lineage는 필요 없습니다. 원본 제공 gene panel의 library sum을 유지합니다.

출력은 `predictions.tsv`의 cell_id/prediction/confidence, 고정 label 순서의 `probabilities.npz`, bundle·입력 hash와 gene coverage가 든 `inference.json`입니다. 확률은 calibration되지 않았습니다. Malignant와 Hepatocyte 정답이 같은 뜻이라고 가정하지 않으며, 이 broad classifier는 악성 여부나 약효를 검증하지 않습니다.


In [2]:
probe = json.loads((REPORT / 'annotation_interface_probe/annotation_query_interface_v2.json').read_text())
assert probe['status'] == 'executed' and not probe['external_data_accessed']
display(pd.DataFrame([{'model': r['model_id'], 'n_real_cells': r['n_cells'], 'representation_cosine_min_vs_cached_development': r.get('representation_roundtrip_cosine_min'), 'input_columns': ', '.join(r['input_columns'])} for r in probe['records']]))


,model,n_real_cells,representation_cosine_min_vs_cached_development,input_columns
0,pca,64,NaN,"cell_id, patient_id, n_counts"
1,scgpt,64,1.0,"cell_id, patient_id, n_counts"
2,geneformer,64,1.0,"cell_id, patient_id, n_counts"
3,geneformer_clcancer,64,1.0,"cell_id, patient_id, n_counts"


In [3]:
rows=[]
for model in ['pca','scgpt','geneformer','geneformer_clcancer']:
    version = 2 if model == 'pca' else 1
    run=json.loads((REPORT/'runs'/f'F00_{model}_development12000_seed42_v{version}/run.json').read_text())
    rows.append({'model':model,'run_id':run['run_id'],'training_cells':run['n_training_cells'],'head_C':run['head_selection']['C'],'class_weight':run['head_selection']['class_weight'],'bundle_sha256':run['bundle_sha256']})
display(pd.DataFrame(rows))


,model,run_id,training_cells,head_C,class_weight,bundle_sha256
0,pca,F00_pca_development12000_seed42_v2,12000,0.01,balanced,c52d02fb34a91160bd6d4980d97ec0c8d185faf7322014...
1,scgpt,F00_scgpt_development12000_seed42_v1,12000,0.01,balanced,446dac0d7b3fc87b4246b46bb87fa7e0a8ba3a1960eaa2...
2,geneformer,F00_geneformer_development12000_seed42_v1,12000,0.01,None,5203f7b7b2cf9ce3ac85c8e6e52cdb9047a6563fcc35df...
3,geneformer_clcancer,F00_geneformer_clcancer_development12000_seed4...,12000,0.01,None,f79bdc07982c1a831550b37ba09dc1f9abdf1512da68e8...


## 선택: 사용자의 새 query 제출

서버에서 실제 count 파일 경로, 새 cache 이름과 결과 디렉터리를 지정한 뒤 아래 옵션을 켭니다. 같은 cache·출력 이름을 재사용해 덮어쓰지 않습니다. raw 데이터·가중치·cell별 예측은 Git에 올리지 않습니다. CPU PCA만 사용할 때는 `predict_annotation.py`를 해당 bundle과 직접 실행할 수도 있습니다.


In [4]:
SUBMIT_NEW_QUERY = False
query_path = Path('/path/to/your/counts.h5ad')
model = 'geneformer'
cache_name = 'my_new_hcc_v1'
output_path = Path('/path/to/new/output_directory')
if SUBMIT_NEW_QUERY:
    assert query_path.is_file() and not output_path.exists()
    result=subprocess.run(['sbatch','--parsable',str(ROOT/'slurm/query_annotation.sh'),str(query_path),model,cache_name,str(output_path)],cwd=ROOT,check=True,text=True,capture_output=True)
    print('submitted:',result.stdout.strip())
else:
    print('실제 저장 결과 읽기 모드: 새 추론 작업을 제출하지 않았습니다.')


실제 저장 결과 읽기 모드: 새 추론 작업을 제출하지 않았습니다.


## 완료된 외부 결과의 해석

GSE151530은 HCC 환자의 알려진 다섯 비악성 세포 계열 전이 평가입니다. 전체50023cells 중 known27212(54.40%), Unknown22811cells입니다. Gold Hepatocyte가 없으므로 fixed-six F1 ceiling은5/6이며 fixed-five는 사전 고정 보조 지표입니다. 모든 셀에 예측을 내는 prediction coverage100%와 gold coverage54.40%를 혼동하지 않습니다. Malignant/unclassified의 lineage 예측은 검증된 악성 판정이나 Hepatocyte 정답이 아닙니다.

아래는 원 frozen batch의 저장 결과를 읽습니다. 새 추론·채점·threshold 선택을 하지 않습니다.


## H5AD categorical metadata의 동등성 확인

H5AD가 gene symbol을 categorical로 저장해도 같은 식별자 값과 발현량을 읽습니다. 실제 개발 세포64개의 metadata 저장 자료형만 바꿔 네 baseline의 예측 확률을 비교했습니다. 이는 생물학적 성능 실험이 아닌 입력 reader 검증입니다.


In [5]:
categorical=json.loads((REPORT/'annotation_interface_probe/annotation_categorical64_v1.json').read_text())
assert categorical['status']=='executed' and categorical['categorical_metadata_storage']
display(pd.DataFrame([{'model':record['model_id'],'probability_max_abs_difference':record['categorical_vs_original_metadata_probability_max_abs_difference']} for record in categorical['records']]))


,model,probability_max_abs_difference
0,pca,0.0
1,scgpt,0.0
2,geneformer,0.0
3,geneformer_clcancer,0.0


In [6]:
verification=json.loads((REPORT/'external_final/completion_verification.json').read_text())
assert verification['status']=='verified_original_completed_batch'
print('Verified original candidates:',verification['n_candidates'])
metrics=pd.read_csv(REPORT/'external_final/metrics.tsv',sep='\t')
display(metrics[metrics.n_training_cells==12000][['candidate','macro_f1','macro_f1_fixed_five','n_cells','n_known','evaluation_coverage','n_unknown_truth','prediction_coverage','ece_10_bins','brier_multiclass']])
print('Primary remains Geneformer; no new model selection or external scoring occurred in this notebook.')


Verified original candidates: 8


,candidate,macro_f1,macro_f1_fixed_five,n_cells,n_known,evaluation_coverage,n_unknown_truth,prediction_coverage,ece_10_bins,brier_multiclass
0,pca,0.809865,0.971838,50023,27212,0.54399,22811,1.0,0.116063,0.059015
1,scgpt,0.825035,0.990042,50023,27212,0.54399,22811,1.0,0.025300,0.012741
2,geneformer,0.817066,0.980479,50023,27212,0.54399,22811,1.0,0.024747,0.027351
3,geneformer_clcancer,0.821433,0.985719,50023,27212,0.54399,22811,1.0,0.022659,0.022541


Primary remains Geneformer; no new model selection or external scoring occurred in this notebook.
